# Projeto: Inserção dos dados na camada bronze do banco de dados.

### Descrição Geral
Este documento descreve o processo de inserção dos dados originados pesquisas em canais de noticias da relação entre a quantidade de chuva necessária para gerar um alerta de alagamento por capitais do Brasil.<br>
O objetivo é fazer a persistência destes dados com o minímo de tratamento possível, somente realizando ações estritamente necessárias.

---

### Informações Técnicas

- **Fonte de Dados**:     
    CEMADEN (Centro Nacional de Monitoramento e Alertas de Desastres Naturais)
    INMET (Instituto Nacional de Meteorologia)
    ANA (Agência Nacional de Águas e Saneamento Básico)
- **Tecnologias Utilizadas**: 
  - Apache Spark (PySpark)
  - Databricks Notebooks

- **Dados persistidos**:
  - Cidade e Unidade Federativa (UF)
  - Data da carga
  - Volume de chuva critico
  - País
---

### Histórico de Atualizações

| Data       | Responsável      | Versão | Descrição                                |
|------------|------------------|--------|--------------------------------------------|
| 17/05/2025 | Lucas Berta      |  1.0   | Criação da inserção dos dados para fonte bronze|


---

In [0]:
%run ../../src/utils/utils_run

In [0]:
%run ./queries_bronze

In [0]:
dict_chuva = enum_configs.VOLUME_CRITICO_CHUVAS.value

In [0]:
dict_capitais = enum_configs.CAPITAIS_BRASIL.value

In [0]:
dados = [(capital, valor) for capital, valor in dict_chuva.items()]

In [0]:
dados_estados = [(sigla, info["capital"]) for sigla, info in dict_capitais.items()]

In [0]:
df_municipios = spark.createDataFrame(dados_estados, ["estado", "municipio"])

In [0]:
schema_df_clima = StructType([
    StructField("municipio", StringType(), True),
    StructField("volume_critico_chuva", StringType(), True)
])

In [0]:
df_chuva = spark.createDataFrame(dados, schema = schema_df_clima)

In [0]:
df = df_chuva.join(df_municipios, on ="municipio", how="left")

In [0]:
df = df.withColumn("pais", lit("Brasil"))

In [0]:
df = adicionaColunasDeCarga(df)

In [0]:
spark.sql(create_table_chuvas)

In [0]:
df.write.saveAsTable(caminho_tabela_chuvas_criticas_bronze, mode="overwrite")